# 90-Day Repurchase Prediction Model
Predicts whether a customer will make another purchase within 90 days, based on their historical transaction behavior.

**Strategy:** Snapshot date = Oct 1, 2024. Features built from Jan–Sep data. Label = did customer transact Oct–Dec?

In [20]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\benja\anaconda3\python.exe
3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]


In [32]:
conda activate base
conda install ipykernel
python -m ipykernel install --user --name base --display-name "Python (base)"

SyntaxError: invalid syntax (1513899221.py, line 1)

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Data/merged_customer_data.csv')

# --- Basic cleanup ---
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce')
df['Engagement Score'] = pd.to_numeric(df['Engagement Score'], errors='coerce')
df['Lifetime Value'] = pd.to_numeric(df['Lifetime Value'], errors='coerce')
df['Data Completeness %'] = pd.to_numeric(df['Data Completeness %'], errors='coerce')
df = df.dropna(subset=['Customer ID', 'Transaction Date', 'Revenue'])
df = df.drop_duplicates()

print(f'Total rows: {df.shape[0]}')
print(f'Unique customers: {df["Customer ID"].nunique()}')
print(f'Date range: {df["Transaction Date"].min().date()} → {df["Transaction Date"].max().date()}')

ImportError: Unable to import required dependencies:
numpy: cannot import name 'set_module' from 'numpy._utils' (unknown location)

## 1. Define Snapshot & Label

In [ ]:
SNAPSHOT_DATE = pd.Timestamp('2024-10-01')  # features built from data BEFORE this date
FUTURE_END    = pd.Timestamp('2024-12-31')  # label window: did they buy in this 90-day period?

past   = df[df['Transaction Date'] <  SNAPSHOT_DATE].copy()
future = df[(df['Transaction Date'] >= SNAPSHOT_DATE) &
            (df['Transaction Date'] <= FUTURE_END)].copy()

future_buyers = set(future['Customer ID'].unique())

print(f'Customers with history before snapshot: {past["Customer ID"].nunique()}')
print(f'Customers who bought Oct–Dec (label=1): {len(future_buyers & set(past["Customer ID"].unique()))}')
print(f'Customers who did NOT (label=0): {past["Customer ID"].nunique() - len(future_buyers & set(past["Customer ID"].unique()))}')

## 2. Feature Engineering
Building 14 features per customer from their Jan–Sep transaction history.

In [ ]:
def avg_purchase_gap(dates):
    """Mean days between consecutive purchases. Returns tenure if only 1 purchase."""
    sorted_dates = sorted(dates)
    if len(sorted_dates) < 2:
        return np.nan
    gaps = [(sorted_dates[i+1] - sorted_dates[i]).days for i in range(len(sorted_dates) - 1)]
    return np.mean(gaps)

features = (
    past.groupby('Customer ID')
    .apply(lambda g: pd.Series({
        # Core RFM
        'Recency':              (SNAPSHOT_DATE - g['Transaction Date'].max()).days,
        'Frequency':            g['Transaction ID'].nunique(),
        'Monetary':             g['Revenue'].sum(),
        'Avg_Order_Value':      g['Revenue'].mean(),
        # Engagement & data quality
        'Engagement_Score':     g['Engagement Score'].max(),
        'Data_Completeness':    g['Data Completeness %'].max(),
        'Lifetime_Value':       g['Lifetime Value'].max(),
        # Behavioral timing
        'Tenure_Days':          (SNAPSHOT_DATE - g['Transaction Date'].min()).days,
        'Avg_Purchase_Gap':     avg_purchase_gap(g['Transaction Date'].tolist()),
        # Recent activity windows
        'Purchases_Last30':     g[g['Transaction Date'] >= SNAPSHOT_DATE - pd.Timedelta(days=30)].shape[0],
        'Purchases_Last60':     g[g['Transaction Date'] >= SNAPSHOT_DATE - pd.Timedelta(days=60)].shape[0],
        'Purchases_Last90':     g[g['Transaction Date'] >= SNAPSHOT_DATE - pd.Timedelta(days=90)].shape[0],
        # Engagement breadth
        'Business_Lines_Engaged': g['Business Lines Engaged'].max(),
        # Categoricals (encoded below)
        'Customer_Data_Type':   g['Customer Data Type'].iloc[0],
        'Region':               g['Region'].iloc[0],
    }))
    .reset_index()
)

# Fill single-purchase gap with tenure (best proxy)
features['Avg_Purchase_Gap'] = features['Avg_Purchase_Gap'].fillna(features['Tenure_Days'])

# Target
features['will_purchase_90d'] = features['Customer ID'].apply(lambda x: int(x in future_buyers))

print(f'Feature matrix: {features.shape}')
print(f"Label distribution:\n{features['will_purchase_90d'].value_counts()}")
features.head()

## 3. Encode Categoricals & Prepare X, y

In [ ]:
from sklearn.preprocessing import LabelEncoder

model_df = features.copy()

# Label-encode low-cardinality categoricals
for col in ['Customer_Data_Type', 'Region']:
    le = LabelEncoder()
    model_df[col + '_enc'] = le.fit_transform(model_df[col].astype(str))
    print(f'{col} classes: {list(le.classes_)}')

FEATURE_COLS = [
    'Recency', 'Frequency', 'Monetary', 'Avg_Order_Value',
    'Engagement_Score', 'Data_Completeness', 'Lifetime_Value',
    'Tenure_Days', 'Avg_Purchase_Gap',
    'Purchases_Last30', 'Purchases_Last60', 'Purchases_Last90',
    'Business_Lines_Engaged',
    'Customer_Data_Type_enc', 'Region_enc'
]

X = model_df[FEATURE_COLS].copy()
y = model_df['will_purchase_90d'].copy()

print(f'\nFeatures used: {len(FEATURE_COLS)}')
print(f'Any nulls? {X.isnull().sum().sum()}')

## 4. Train/Test Split & Class Imbalance Handling

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np


np.random.seed(42)

idx_1 = np.where(y == 1)[0]
idx_0 = np.where(y == 0)[0]

# 80/20 split within each class
n_test_1 = int(len(idx_1) * 0.2)
n_test_0 = int(len(idx_0) * 0.2)

np.random.shuffle(idx_1)
np.random.shuffle(idx_0)

test_idx  = np.concatenate([idx_1[:n_test_1], idx_0[:n_test_0]])
train_idx = np.concatenate([idx_1[n_test_1:], idx_0[n_test_0:]])

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'Train label dist:\n{y_train.value_counts()}')
print(f'Test label dist:\n{y_test.value_counts()}')

In [ ]:
import sys
!C:\Users\benja\anaconda3\python.exe -m pip install -U numpy scikit-learn

## 5. Train Model — Random Forest with Class Balancing
`class_weight='balanced'` compensates for the ~75/25 label imbalance automatically.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['No Purchase (0)', 'Will Purchase (1)']))
print(f'ROC AUC:  {roc_auc_score(y_test, y_prob):.3f}')
print(f'PR  AUC:  {average_precision_score(y_test, y_prob):.3f}')

## 6. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=ax,
    display_labels=['No Purchase', 'Will Purchase'],
    cmap='Blues'
)
ax.set_title('Confusion Matrix — 90-Day Repurchase')
plt.tight_layout()
plt.show()

## 7. ROC & Precision-Recall Curves

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0])
axes[0].set_title('ROC Curve')

PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

## 8. Cross-Validation (5-Fold Stratified)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    rf, X, y, cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score=False
)

print('5-Fold CV Results:')
for metric in ['test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    label = metric.replace('test_', '').capitalize()
    print(f'  {label:<12} mean={cv_results[metric].mean():.3f}  std={cv_results[metric].std():.3f}')

## 9. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Random Forest Feature Importance — 90-Day Repurchase')
ax.set_ylabel('Importance')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print(importances)

## 10. SHAP Explainability

In [ ]:
import shap

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

# Global bar chart
shap.summary_plot(shap_values[:, :, 1], X, feature_names=FEATURE_COLS, plot_type='bar')

In [ ]:
# Direction of effects (beeswarm)
shap.summary_plot(shap_values[:, :, 1], X, feature_names=FEATURE_COLS)

## 11. Score All Customers — Priority List

In [ ]:
model_df['repurchase_probability'] = rf.predict_proba(X)[:, 1]
model_df['repurchase_prediction']  = rf.predict(X)

# Risk tier
def risk_tier(p):
    if p >= 0.75: return 'High Likelihood'
    elif p >= 0.50: return 'Moderate'
    elif p >= 0.30: return 'Low'
    else: return 'Unlikely'

model_df['repurchase_tier'] = model_df['repurchase_probability'].apply(risk_tier)

output_cols = [
    'Customer ID', 'repurchase_probability', 'repurchase_tier',
    'Recency', 'Frequency', 'Monetary', 'Purchases_Last90',
    'Engagement_Score', 'Lifetime_Value', 'Customer_Data_Type', 'Region'
]

priority_list = (
    model_df[output_cols]
    .sort_values('repurchase_probability', ascending=False)
    .reset_index(drop=True)
)

print('Tier distribution:')
print(model_df['repurchase_tier'].value_counts())
print()
priority_list.head(15)

## 12. Revenue at Stake by Tier

In [ ]:
tier_summary = (
    model_df.groupby('repurchase_tier')
    .agg(
        Customers=('Customer ID', 'count'),
        Avg_Repurchase_Prob=('repurchase_probability', 'mean'),
        Total_LTV=('Lifetime_Value', 'sum')
    )
    .sort_values('Avg_Repurchase_Prob', ascending=False)
    .round(3)
)

print(tier_summary.to_string())

unlikely_ltv = model_df.loc[model_df['repurchase_tier'] == 'Unlikely', 'Lifetime_Value'].sum()
print(f'\nLTV at risk (Unlikely tier): ${unlikely_ltv:,.0f}')
print(f'If 50% are retained: ${unlikely_ltv * 0.5:,.0f}')

## 13. Export Priority List to CSV

In [ ]:
priority_list.to_csv('repurchase_priority_list.csv', index=False)
print('Saved: repurchase_priority_list.csv')
priority_list.head()